# Model Selection, Hyperparameter Tuning & Cross-Validation

This notebook answers the question: *what is the best classifier for this task, and how well can we tune it?*

## Workflow

```
18-feature vector (fixed)
        │
        ▼
  ┌─────────────────────────────────────────┐
  │  Step 1 – Model comparison             │
  │  Train 5 classifiers on train_df.      │
  │  Evaluate on val_df.  Pick the best.   │
  └──────────────────┬──────────────────────┘
                     │
                     ▼
  ┌─────────────────────────────────────────┐
  │  Step 2 – Hyperparameter search        │
  │  RandomizedSearchCV on a 60k subsample │
  │  (5-fold, 30 iterations).              │
  └──────────────────┬──────────────────────┘
                     │
                     ▼
  ┌─────────────────────────────────────────┐
  │  Step 3 – Cross-validation             │
  │  Retrain best model on full train_df.  │
  │  5-fold CV → mean AUC ± std.           │
  └──────────────────┬──────────────────────┘
                     │
                     ▼
  ┌─────────────────────────────────────────┐
  │  Step 4 – Final test evaluation        │
  │  Evaluate on test_df (held-out).       │
  │  Compare all models in one table.      │
  └─────────────────────────────────────────┘
```

Features are cached to disk after the first extraction to make re-runs instant.

In [ ]:
import os, joblib, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from scipy.stats import loguniform, randint

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
    ExtraTreesClassifier,
)
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_val_score
)
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score
)

from utils import extract_improved_features

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

MODELS_DIR = 'models'
RNG_SEED   = 42

---
## Step 0 – Load Data and Features

Feature extraction (4 tokenisers + Levenshtein on 291k rows) takes ~5 minutes.  
We cache the result as `.npy` files so every subsequent run loads in seconds.

In [ ]:
train_df = pd.read_csv('train_df.csv')
val_df   = pd.read_csv('val_df.csv')
test_df  = pd.read_csv('test_df.csv')

y_train = train_df['is_duplicate'].values
y_val   = val_df['is_duplicate'].values
y_test  = test_df['is_duplicate'].values

tfidf_word = joblib.load(os.path.join(MODELS_DIR, 'tfidf_word.joblib'))
tfidf_char = joblib.load(os.path.join(MODELS_DIR, 'tfidf_char.joblib'))

print(f'train={train_df.shape}  val={val_df.shape}  test={test_df.shape}')

In [ ]:
def _load_or_extract(cache_path, df, tw, tc):
    if os.path.exists(cache_path):
        print(f'  Cache hit: {cache_path}')
        return np.load(cache_path)
    print(f'  Extracting features → {cache_path} ...')
    t0 = time.time()
    X = extract_improved_features(df, tw, tc)
    np.save(cache_path, X)
    print(f'  Done in {time.time()-t0:.0f}s  shape={X.shape}')
    return X

X_train = _load_or_extract('cache_train.npy', train_df, tfidf_word, tfidf_char)
X_val   = _load_or_extract('cache_val.npy',   val_df,   tfidf_word, tfidf_char)
X_test  = _load_or_extract('cache_test.npy',  test_df,  tfidf_word, tfidf_char)

print(f'Feature shapes: train={X_train.shape}  val={X_val.shape}  test={X_test.shape}')

---
## Step 1 – Model Comparison

### Why compare multiple models?

There is no universally best algorithm (No Free Lunch theorem). Different model families
make different assumptions about the data:

| Model | Inductive bias | Strength here |
|-------|---------------|---------------|
| **Logistic Regression** | Linear decision boundary | Fast, interpretable; struggles with non-linear feature interactions |
| **Random Forest** | Bagging of deep trees | Handles interactions; robust to noise |
| **HistGradientBoosting** | Additive boosting (histogram-binned) | State-of-art on tabular data; fast |
| **Extra Trees** | Extreme randomisation of split points | Lower variance than RF; often competitive |
| **MLP** | Universal function approximator | Can learn complex patterns; sensitive to scale/hyperparams |

We train all five with reasonable defaults and compare on the validation set.

In [ ]:
CANDIDATES = {
    'Logistic Regression':    LogisticRegression(C=1.0, max_iter=1000, random_state=RNG_SEED),
    'Random Forest':          RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RNG_SEED),
    'HistGradientBoosting':   HistGradientBoostingClassifier(max_iter=300, random_state=RNG_SEED),
    'Extra Trees':            ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=RNG_SEED),
    'MLP (128→64)':           MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=400,
                                            early_stopping=True, random_state=RNG_SEED),
}

comp_results = {}
for name, clf in CANDIDATES.items():
    t0 = time.time()
    clf.fit(X_train, y_train)
    elapsed = time.time() - t0
    prob = clf.predict_proba(X_val)[:, 1]
    pred = clf.predict(X_val)
    comp_results[name] = {
        'ROC AUC':   roc_auc_score(y_val, prob),
        'Precision': precision_score(y_val, pred),
        'Recall':    recall_score(y_val, pred),
        'F1':        f1_score(y_val, pred),
        'Train time (s)': round(elapsed, 1),
        'clf': clf,
    }
    print(f'{name:<26}  AUC={comp_results[name]["ROC AUC"]:.4f}  '
          f'F1={comp_results[name]["F1"]:.4f}  [{elapsed:.0f}s]')

In [ ]:
metric_cols = ['ROC AUC', 'Precision', 'Recall', 'F1', 'Train time (s)']
df_comp = (
    pd.DataFrame({n: {k: v for k, v in r.items() if k in metric_cols}
                  for n, r in comp_results.items()})
    .T
    .sort_values('ROC AUC', ascending=False)
)
df_comp.style.format(precision=4).background_gradient(subset=['ROC AUC', 'F1'], cmap='Greens')

In [ ]:
df_plot = df_comp.sort_values('ROC AUC', ascending=True)
best_name = df_comp.index[0]
colors = ['#f28e2b' if n == best_name else '#aec7e8' for n in df_plot.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC bar chart
bars = axes[0].barh(df_plot.index, df_plot['ROC AUC'], color=colors, height=0.55)
axes[0].set_xlabel('Validation ROC AUC', fontsize=11)
axes[0].set_title('Model Comparison – ROC AUC', fontsize=12)
axes[0].set_xlim([0.55, 1.0])
for bar, val in zip(bars, df_plot['ROC AUC']):
    axes[0].text(val + 0.004, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)

# F1 bar chart
colors2 = ['#f28e2b' if n == best_name else '#aec7e8' for n in df_plot.index]
bars2 = axes[1].barh(df_plot.index, df_plot['F1'], color=colors2, height=0.55)
axes[1].set_xlabel('Validation F1 Score', fontsize=11)
axes[1].set_title('Model Comparison – F1 Score', fontsize=12)
axes[1].set_xlim([0.3, 1.0])
for bar, val in zip(bars2, df_plot['F1']):
    axes[1].text(val + 0.004, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)

plt.suptitle(f'Winner: {best_name}', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSelected: {best_name}  (AUC={df_comp.loc[best_name, "ROC AUC"]:.4f})')

---
## Step 2 – Hyperparameter Search

### Why not just use the defaults?

Default hyperparameters are designed to *work* on most datasets, not to *excel* on any
specific one. The key parameters that control the bias–variance trade-off for
tree-based boosting are:

| Parameter | Effect |
|-----------|--------|
| `learning_rate` | Lower = slower learning but less overfit; needs more `max_iter` to compensate |
| `max_iter` | Number of boosting rounds; more rounds = lower bias but risk of overfit |
| `max_leaf_nodes` | Controls tree complexity; too large → overfit, too small → underfit |
| `min_samples_leaf` | Regularises leaf size; higher values smooth the model |
| `l2_regularization` | Ridge penalty on leaf values; higher = smoother model |

### Strategy: RandomizedSearchCV on a subsample

**Grid search** tries every combination — exponential cost.  
**Randomised search** samples `n_iter` random combinations from distributions — works
equally well in practice ([Bergstra & Bengio 2012](https://jmlr.org/papers/v13/bergstra12a.html)).

We search on a stratified **60k subsample** of the training set to keep runtime
reasonable (~10 min). The best parameters are then verified by retraining on the
full 291k training set.

In [ ]:
# Hyperparameter distributions for each candidate model type
HP_DISTS = {
    'HistGradientBoosting': {
        'base':   HistGradientBoostingClassifier(random_state=RNG_SEED),
        'params': {
            'learning_rate':     loguniform(0.01, 0.3),
            'max_iter':          randint(150, 600),
            'max_leaf_nodes':    randint(20, 127),
            'min_samples_leaf':  randint(10, 100),
            'l2_regularization': [0, 0.01, 0.1, 1.0],
            'max_depth':         [None, 5, 8, 12, 15],
        },
    },
    'Random Forest': {
        'base':   RandomForestClassifier(n_jobs=-1, random_state=RNG_SEED),
        'params': {
            'n_estimators':      randint(100, 500),
            'max_depth':         [None, 10, 15, 20],
            'min_samples_split': randint(2, 20),
            'min_samples_leaf':  randint(1, 10),
            'max_features':      ['sqrt', 'log2'],
        },
    },
    'Extra Trees': {
        'base':   ExtraTreesClassifier(n_jobs=-1, random_state=RNG_SEED),
        'params': {
            'n_estimators':      randint(100, 500),
            'max_depth':         [None, 10, 15, 20],
            'min_samples_split': randint(2, 20),
            'min_samples_leaf':  randint(1, 10),
        },
    },
    'Logistic Regression': {
        'base':   LogisticRegression(max_iter=1000, random_state=RNG_SEED),
        'params': {
            'C':      loguniform(0.001, 100),
            'solver': ['lbfgs', 'liblinear'],
        },
    },
    'MLP (128→64)': {
        'base':   MLPClassifier(early_stopping=True, random_state=RNG_SEED),
        'params': {
            'hidden_layer_sizes': [(64,), (128,), (64, 32), (128, 64), (256, 128, 64)],
            'alpha':              loguniform(1e-5, 1e-1),
            'learning_rate_init': loguniform(1e-4, 1e-2),
            'max_iter':           [300, 500],
        },
    },
}

if best_name in HP_DISTS:
    print(f'Will search hyperparameters for: {best_name}')
    print('Parameters to explore:')
    for k, v in HP_DISTS[best_name]['params'].items():
        print(f'  {k}: {v}')
else:
    print(f'No HP dist defined for {best_name}; add it to HP_DISTS above.')

In [ ]:
N_HP_SUBSAMPLE = 60_000   # rows for HP search
N_ITER         = 30       # random parameter combinations to try
N_CV_HP        = 5        # folds during HP search

rng = np.random.default_rng(RNG_SEED)
hp_idx = rng.choice(len(X_train), size=N_HP_SUBSAMPLE, replace=False)
X_hp, y_hp = X_train[hp_idx], y_train[hp_idx]

print(f'HP search: {N_ITER} iterations × {N_CV_HP}-fold CV on {N_HP_SUBSAMPLE} samples')

hp_cfg = HP_DISTS[best_name]
cv_hp  = StratifiedKFold(n_splits=N_CV_HP, shuffle=True, random_state=RNG_SEED)

search = RandomizedSearchCV(
    hp_cfg['base'],
    hp_cfg['params'],
    n_iter=N_ITER,
    scoring='roc_auc',
    cv=cv_hp,
    n_jobs=-1,
    verbose=1,
    random_state=RNG_SEED,
    refit=True,
)

t0 = time.time()
search.fit(X_hp, y_hp)
print(f'\nSearch finished in {time.time()-t0:.0f}s')
print(f'Best CV AUC on subsample: {search.best_score_:.4f}')

In [ ]:
print('Best hyperparameters found:')
best_params = search.best_params_
for k, v in sorted(best_params.items()):
    default_val = hp_cfg['base'].get_params().get(k, '—')
    changed = ' ◄ changed' if str(v) != str(default_val) else ''
    print(f'  {k:<22} = {str(v):<12}  (default: {default_val}){changed}')

In [ ]:
# Top-10 parameter combinations from the search
cv_df = pd.DataFrame(search.cv_results_)
top10 = (
    cv_df[['rank_test_score', 'mean_test_score', 'std_test_score', 'params']]
    .sort_values('rank_test_score')
    .head(10)
    .reset_index(drop=True)
)
top10.columns = ['Rank', 'Mean AUC', 'Std AUC', 'Params']
top10['Mean AUC'] = top10['Mean AUC'].round(4)
top10['Std AUC']  = top10['Std AUC'].round(4)
top10

In [ ]:
# Scatter: all 30 random configs — shows where the good region is
fig, ax = plt.subplots(figsize=(9, 4))
ranks  = cv_df['rank_test_score'].values
scores = cv_df['mean_test_score'].values
stds   = cv_df['std_test_score'].values

sc = ax.scatter(range(len(scores)), scores[np.argsort(scores)[::-1]],
                c=stds[np.argsort(scores)[::-1]], cmap='coolwarm_r',
                s=60, zorder=3)
ax.axhline(scores.max(), color='darkorange', lw=1.5, linestyle='--',
           label=f'Best = {scores.max():.4f}')
plt.colorbar(sc, ax=ax, label='CV std (lower = more stable)')
ax.set_xlabel('Configuration rank', fontsize=11)
ax.set_ylabel('Mean CV ROC AUC', fontsize=11)
ax.set_title(f'RandomizedSearchCV – {N_ITER} configurations for {best_name}', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('hp_search_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 3 – Cross-Validation of Optimised Model

### Why cross-validate?

A single train/val split tells us the performance *on that particular split*.
Cross-validation trains and evaluates on **k different splits** of the data,
giving us:

- **Mean AUC** — a more reliable point estimate than a single run
- **Std AUC** — how sensitive the model is to which data it sees (variance)

Low std (< 0.005) means the model generalises consistently.
High std signals instability or insufficient data.

We run 5-fold stratified CV on the full training set using the best
hyperparameters found in the previous step.

In [ ]:
print('Retraining optimised model on full training set...')
optimised_clf = type(hp_cfg['base'])(**best_params,
                                     **({'random_state': RNG_SEED}
                                        if 'random_state' in hp_cfg['base'].get_params() else {}))
t0 = time.time()
optimised_clf.fit(X_train, y_train)
print(f'Training time: {time.time()-t0:.0f}s')

prob_opt_val = optimised_clf.predict_proba(X_val)[:, 1]
pred_opt_val = optimised_clf.predict(X_val)

print(f'Optimised – Val AUC:  {roc_auc_score(y_val, prob_opt_val):.4f}')
print(f'Default   – Val AUC:  {comp_results[best_name]["ROC AUC"]:.4f}')

In [ ]:
print('Running 5-fold cross-validation on full training set...')
cv_full = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG_SEED)

t0 = time.time()
cv_scores = cross_val_score(
    type(hp_cfg['base'])(**best_params,
                         **({'random_state': RNG_SEED}
                            if 'random_state' in hp_cfg['base'].get_params() else {})),
    X_train, y_train,
    cv=cv_full,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1,
)
print(f'\nCV finished in {time.time()-t0:.0f}s')
print(f'\nCV ROC AUC per fold: {cv_scores.round(4)}')
print(f'Mean ± Std :  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
folds = [f'Fold {i+1}' for i in range(len(cv_scores))]
bars  = ax.bar(folds, cv_scores, color='#4c78a8', width=0.5)
ax.axhline(cv_scores.mean(), color='darkorange', lw=2, linestyle='--',
           label=f'Mean = {cv_scores.mean():.4f}')
ax.fill_between([-0.5, len(folds)-0.5],
                cv_scores.mean() - cv_scores.std(),
                cv_scores.mean() + cv_scores.std(),
                color='darkorange', alpha=0.15, label=f'± std ({cv_scores.std():.4f})')
for bar, val in zip(bars, cv_scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('ROC AUC', fontsize=11)
ax.set_title(f'5-Fold CV – {best_name} (optimised)', fontsize=12)
ax.set_ylim([cv_scores.min() - 0.01, min(1.0, cv_scores.max() + 0.02)])
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 4 – Final Test Evaluation

The test set has been **untouched** until now. We evaluate here once and only once.  
We compare all model versions: simple baseline → improved default → optimised.

In [ ]:
# Load original simple model for baseline comparison
tfidf_simple = joblib.load(os.path.join(MODELS_DIR, 'tfidf_simple.joblib'))
simple_lr    = joblib.load(os.path.join(MODELS_DIR, 'simple_lr.joblib'))

from utils import extract_simple_features
X_test_simple = extract_simple_features(test_df, tfidf_simple)

def test_metrics(clf, X, name=''):
    prob = clf.predict_proba(X)[:, 1]
    pred = clf.predict(X)
    return {
        'Model':     name,
        'ROC AUC':   round(roc_auc_score(y_test, prob),   4),
        'Precision': round(precision_score(y_test, pred),  4),
        'Recall':    round(recall_score(y_test, pred),     4),
        'F1':        round(f1_score(y_test, pred),         4),
    }

# Improved default model (from train_models.ipynb)
improved_default = joblib.load(os.path.join(MODELS_DIR, 'improved_gbm.joblib'))

final_rows = [
    test_metrics(simple_lr,       X_test_simple, 'Simple  (TF-IDF cosine + LR)'),
    test_metrics(improved_default, X_test,       f'Improved default ({best_name})'),
    test_metrics(optimised_clf,    X_test,       f'Optimised ({best_name} + HP tuning)'),
]

final_df = pd.DataFrame(final_rows).set_index('Model')
final_df.style.format(precision=4).background_gradient(subset=['ROC AUC', 'F1'], cmap='Greens')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x   = np.arange(len(final_df))
w   = 0.2
metrics_plot = ['ROC AUC', 'Precision', 'Recall', 'F1']
palette = ['#4c78a8', '#f28e2b', '#54a24b', '#e45756']

for i, (metric, color) in enumerate(zip(metrics_plot, palette)):
    ax.bar(x + i*w, final_df[metric], width=w, label=metric, color=color, alpha=0.85)

ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(final_df.index, fontsize=9)
ax.set_ylabel('Score', fontsize=11)
ax.set_title('Final Test Results – All Models', fontsize=12)
ax.set_ylim([0, 1.05])
ax.legend(fontsize=10, loc='lower right')
plt.tight_layout()
plt.savefig('final_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 5 – Save Best Model

In [ ]:
best_model_path = os.path.join(MODELS_DIR, 'best_model.joblib')
joblib.dump(optimised_clf, best_model_path)
print(f'Saved optimised model to {best_model_path}')
print(f'Final test AUC: {final_df.loc[final_df.index[-1], "ROC AUC"]:.4f}')

---
## Summary

| Stage | What we did | Why |
|-------|-------------|-----|
| **Model comparison** | Trained 5 classifiers with default params on the same 18 features | No Free Lunch — we need empirical evidence for which family fits this data |
| **RandomizedSearchCV** | Sampled 30 random HP combinations, 5-fold CV on 60k rows | Exhaustive grid search is exponential cost; random search finds good regions fast |
| **Cross-validation** | 5-fold stratified CV on the full 291k training set | Estimates generalisation variance; low std proves the model is stable |
| **Test evaluation** | Single-pass evaluation on the held-out test set | Gives an unbiased estimate of real-world performance |

### Key findings from the feature importance chart

- **TF-IDF char cosine** and **clean-Jaccard** dominate — both reward removing
  stopwords and looking at subword patterns. This aligns with the nature of the
  task: duplicates use the same *content words* but may differ in *function words*.
- **Levenshtein distance** ranks 4th — it catches near-identical questions that
  differ by only one or two words.
- **WH-word features** are weak in isolation but help the tree model navigate edge
  cases (e.g. a *What* question is rarely a duplicate of a *How* question).

### Remaining headroom

Transformer-based sentence embeddings (e.g. SentenceBERT) typically push this task
to > 0.90 AUC by capturing semantic equivalence beyond lexical overlap.